# Generowanie danych do plików csv

## 1 Użyte biblioteki

In [134]:
from faker import Faker
import random
from countryinfo import CountryInfo
import csv
import string
import unicodedata
from datetime import datetime, timedelta

In [103]:
def remove_special_characters(text):
    normalized_string = unicodedata.normalize('NFD', text)
    final_string = ''.join(
        char for char in normalized_string if ord(char) <= 127 and (char.isalnum() or char.isspace())
    )
    return final_string

## 2 Generowanie informacji o ludziach z podziałem na studentów i pracowników

Generator gwarantuje unikatowość identyfikatotów, adresów email oraz numerów telefonów. Obecna implementacja umożliwia generowanie danych dla różnych krajów.

### 2.1 Klasa Person

In [104]:
class Person:
    
    student_counter = 1000
    employee_counter = 1000
    translator_counter = 100
    generated_phone_numbers = set()
    generated_emails = set()
    
    def __init__(self, position, symbol):
        self.firstNameGenerate(symbol)
        self.lastNameGenerate(symbol)
        self.status = position
        self.birthDateGenerate()
        self.cityGenerate(symbol)
        self.streetAddressGenerate(symbol)
        self.getCountryName(symbol)
        self.getStudentID()
        self.getEmployeeID()
        self.getTranslatorID()
        self.phoneNumberGenerate(symbol)
        self.emailGenerate()
        self.translator_languages()
        self.jobPosition = None
    
    def firstNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.firstName = fake.first_name()
    
    def lastNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.lastName = fake.last_name()
        
    def cityGenerate(self, symbol):
        fake = Faker(symbol)
        self.city = fake.city()
    
    def streetAddressGenerate(self, symbol):
        fake = Faker(symbol)
        self.streetAddress = fake.street_address()
    
    def getCountryName(self, symbol):
        
        country_dict = {
            "US": "Stany Zjednoczone",
            "PL": "Polska",
            "DE": "Niemcy",
            "GB": "Wielka Brytania",
            "FR": "Francja",
            "IT": "Włochy"
        }
        
        country_code = symbol.split('_')[1]
        self.country = country_dict[country_code]
    
    
    def birthDateGenerate(self):
        position = self.status
        fake = Faker()
        match position:
            case 'student':
                age_ranges = [(18, 25), (26, 30), (31, 50), (51, 65)]
                weights = [0.6, 0.2, 0.15, 0.05]
            case 'employee':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.2, 0.4, 0.3, 0.1]
            case 'translator':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.4, 0.3, 0.2, 0.1]
                
            
        selected_range = random.choices(age_ranges, weights=weights, k=1)[0]
        min_age, max_age = selected_range
        self.birthDate = fake.date_of_birth(None, min_age, max_age)
    
    def getStudentID(self):
        if self.status == 'student':
            self.studentID = Person.student_counter
            Person.student_counter += 1
    
    def getEmployeeID(self):
        if self.status == 'employee':
            self.employeeID = Person.employee_counter
            Person.employee_counter += 1
    
    def getTranslatorID(self):
        if self.status == 'translator':
            self.translatorID = Person.translator_counter
            Person.translator_counter += 1
    
    def phoneNumberGenerate(self, symbol):
        faker = Faker(symbol)
        country_code = symbol.split('_')[1]
        informations = CountryInfo(country_code)
        calling_code = informations.calling_codes()[0]
        
        while True:
            phone_number = faker.phone_number()
            if not phone_number[0] == '+':
                phone_number = "+" + calling_code + " " + phone_number
            if phone_number not in Person.generated_phone_numbers:
                self.phone = phone_number
                Person.generated_phone_numbers.add(phone_number)
                break
    
    def emailGenerate(self):
        domains = [
            "gmail.com",
            "outlook.com",
            "interia.pl",
            "yahoo.com",
            "wp.pl"
        ]
        weights = [0.4, 0.1, 0.2, 0.1, 0.2]
        trans = str.maketrans("ąćęłńóśźż", "acelnoszz")
        first_name = remove_special_characters(self.firstName)
        last_name = remove_special_characters(self.lastName)
        
        while True:
            domain =  random.choices(domains, weights=weights, k=1)[0]
            random_number = random.randint(1, 9999)
            random_separator = random.choice([".", "_", "-"])
            
            email_prefix = random.choice([
            f"{first_name}{random_number}{last_name}",
            f"{first_name}{random_separator}{last_name}",
            f"{last_name}{random_separator}{first_name}",
            f"{last_name}{random_separator}{first_name}{random_number}",
            f"{first_name}{random_number}"
            f"{last_name}{random_separator}{random_number}"
            ]).lower()
            
            email = email_prefix + "@" + domain
            if email not in Person.generated_emails:
                self.email = email
                Person.generated_emails.add(email)
                break
            
    def translator_languages(self):
        if self.status == 'translator':
            languagesIDs = [1, 2, 3, 4, 5]
            numbers_of_languages = random.randint(1, 3)
            self.languagesID = random.sample(languagesIDs, k=numbers_of_languages)

### 2.2 Przykład użycia

In [105]:

person = Person('employee', 'pl_PL')
print('First name:', person.firstName)
print('Last name:', person.lastName)
print('Position:', person.status)
print('Birth Date:', person.birthDate)
print('City:', person.city)
print('Address:', person.streetAddress)
print('Country:', person.country)
print('Employee ID:', person.employeeID)
print('Phone number:', person.phone)
print('Email:', person.email)


First name: Igor
Last name: Gwara
Position: employee
Birth Date: 1990-08-09
City: Bartoszyce
Address: al. Wyszyńskiego 74
Country: Polska
Employee ID: 1000
Phone number: +48 517 942 573
Email: igor574gwara.574@wp.pl


### 2.3 Zapisywanie danych do pliku .csv

In [106]:
def SavetoCsv(filename, data):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        for row in data:
            writer.writerow(row)

### 2.4 Generowanie wszystkich studentów, pracowników i tłumaczy

In [107]:
students = []
employees = []
translators = []
for i in range(100):
    student = Person('student', 'pl_PL')
    students.append(student)
for i in range(20):
    employee = Person('employee', 'pl_PL')
    employees.append(employee)
for i in range(10):
    translator = Person('translator', 'pl_PL')
    translators.append(translator)

### 2.5 Zapisywanie danych dla studentów do csv

In [108]:
students_info = []
for i in range(len(students)):
    student = students[i]
    student_data = [
        student.studentID, 
        student.firstName, 
        student.lastName, 
        student.birthDate, 
        student.country, 
        student.city, 
        student.streetAddress, 
        student.email, 
        student.phone
    ]
    students_info.append(student_data)
SavetoCsv('student.csv', students_info)

### 2.6 Zapisywanie danych dla pracowników do csv

In [109]:
employees_info = []
for i in range(len(employees)):
    employee = employees[i]
    employee_data = [
        employee.employeeID, 
        employee.firstName, 
        employee.lastName, 
        employee.birthDate, 
        employee.country, 
        employee.city, 
        employee.streetAddress, 
        employee.email, 
        employee.phone
    ]
    employees_info.append(employee_data)
SavetoCsv('employee.csv', employees_info)

### 2.7 Zapisywanie danych dla tłumaczy do pliku

In [110]:
translators_info = []
for i in range(len(translators)):
    translator = translators[i]
    translator_data = [
        translator.translatorID, 
        translator.firstName, 
        translator.lastName, 
        translator.birthDate, 
        translator.country, 
        translator.city, 
        translator.streetAddress, 
        translator.email, 
        translator.phone
    ]
    translators_info.append(translator_data)
SavetoCsv('translator.csv', translators_info)

### 2.8 Mapa języków

In [117]:
languages = {
    1 : 'Angielski',
    2 : 'Hiszpański',
    3 : 'Francuski',
    4 : 'Niemiecki',
    5 : 'Włoski'
}

### 2.9 Zapisywanie języków do csv

In [120]:
languages_info = []
for translator in translators:
    for language in translator.languagesID:
        languages_info.append([language, languages[language], translator.translatorID])
SavetoCsv('languages.csv', languages_info)

## 3 Generowanie informacji na temat lokalizacji zajęć

In [78]:
class LectureRoom:
    id_counter = 100
    lecture_rooms = set()
    
    def __init__(self):
        self.id = LectureRoom.id_counter
        self.buildingInfoGenerate()
        LectureRoom.id_counter += 1
    
    def buildingInfoGenerate(self):
        buildings = ['A-0', 'A-1', 'A-2', 'B-1', 'B-2', 'C-1', 'C-2', 'C-3']
        while True:
            floor = random.randint(1, 4)
            class_number = random.randint(1, 20)
            building = random.choice(buildings)
            new_class = str(class_number) + " " + str(floor) + building
            
            if new_class not in LectureRoom.lecture_rooms:
                self.building = building
                self.floor = floor
                self.classNumber = class_number
                LectureRoom.lecture_rooms.add(new_class)
                break

### 3.1 Przykład generowanych sal lekcyjnych

In [79]:
classes = [LectureRoom(), LectureRoom(), LectureRoom()]

for i in range(3):
    print(f"\nSala {i}")
    print('Room ID:', classes[i].id)
    print('Building:', classes[i].building)
    print('Floor:', classes[i].floor)
    print('Class number:', classes[i].classNumber)


Sala 0
Room ID: 100
Building: B-1
Floor: 2
Class number: 6

Sala 1
Room ID: 101
Building: C-3
Floor: 1
Class number: 9

Sala 2
Room ID: 102
Building: C-3
Floor: 3
Class number: 15


### 3.2 Przykładowe generowanie danych dla sal i zapisanie ich

In [80]:
rooms_info = []
for i in range(100):
    room = LectureRoom()
    room_data = [
        room.id,
        room.building, 
        room.floor, 
        room.classNumber
    ]
    rooms_info.append(room_data)
SavetoCsv('lecturerooms.csv', rooms_info)

## 4 Generowanie webinarów

W celu uniknięcia w przyszlości kolizji wynikającej z przypisania jednemu pracownikowi dwóch zajęć jednoczeńsnie, wszystkie daty wraz z ID pracownika będą zapisywane w jednym set.

In [121]:
date_employee = set()

In [135]:
class Webinar:
    
    webinar_counter = 1000
    generated_links = set()
    
    def __init__(self):
        self.getID()
        self.linkGenerate()
        self.priceGenerate()
        self.getTranslator()
        self.getEmployee()
        self.dateGenerate()
        
        
    def getID(self):
        self.id = Webinar.webinar_counter
        Webinar.webinar_counter += 1
    
    def linkGenerate(self):
        prefix = "https://teams.microsoft.com/l/meetup-join/"
        while True:
            sufix = random.choices(string.ascii_lowercase + string.digits, k = 20)
            sufix = ''.join(sufix)
            if sufix not in Webinar.generated_links:
                link = prefix + sufix
                self.link = link
                Webinar.generated_links.add(link)
                break
    
    def priceGenerate(self):
        prices = [29.99, 99, 149, 249]
        weights = [0.3, 0.5, 0.15, 0.05]
        self.price = random.choices(prices, weights=weights, k=1)[0]
    
    def getTranslator(self):
        translator = random.choice(translators)
        self.translatorID = translator.translatorID
        self.languageID = random.choice(translator.languagesID)
    
    def getEmployee(self):
        employee = random.choice(employees)
        self.employeeID = employee.employeeID    
    
    def dateGenerate(self):
        fake = Faker()
        teacher = self.employeeID
        hours = [
            '15:00',
            '16:45',
            '18:30',
            '20:15'
        ]
        while True:
            hour = random.choice(hours)
            random_days = random.randint(0, 90)    
            date = datetime.now() + timedelta(days=random_days)

            random_datetime_str = f"{date.strftime('%Y-%m-%d')} {hour}"
            random_datetime = datetime.strptime(random_datetime_str, '%Y-%m-%d %H:%M')
            
            if random_datetime not in date_employee:
                self.date = random_datetime
                date_employee.add(random_datetime)
                break

### 3.1 Przykładowy wygenerowany webinar

In [144]:
webinar = Webinar()
print('Webinar ID:', webinar.id)
print('Date and time:', webinar.date)
print('Employee ID:', webinar.employeeID)
print('Translator ID:', webinar.translatorID)
print('Language ID:', webinar.languageID)
print('Link:', webinar.link)

Webinar ID: 1008
Date and time: 2024-12-31 18:30:00
Employee ID: 1013
Translator ID: 103
Language ID: 2
Link: https://teams.microsoft.com/l/meetup-join/1epal31lhglfaq7dgq40
